In [2]:
!pip install sentencepiece sacrebleu evaluate datasets
!pip install -U transformers==4.44.2 peft==0.10.0 accelerate==0.33.0 bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 33.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow

In [3]:
from peft import LoraConfig, get_peft_model
import os
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
from transformers import TrainerCallback
import time
import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"


2025-11-02 03:32:23.069065: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762054343.264688      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762054343.322204      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
dataset = load_dataset("csv", data_files="/kaggle/input/d/muaadhnazly/sin-eng/sin-eng.csv")

dataset = dataset["train"].train_test_split(test_size=0.2)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-1.3B") 
model = AutoModelForSeq2SeqLM.from_pretrained( "facebook/nllb-200-1.3B", 
        torch_dtype=torch.float32, 
        device_map="cuda")
model.to('cuda')


Generating train split: 0 examples [00:00, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-23): 24 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=8192, bias=True)
          (fc2): Linear(in_features=8192, out_features=1024, bias=True)
       

In [8]:
source_lang = "sin_Sinh"
target_lang = "eng_Latn"
tokenizer.src_lang = source_lang
tokenizer.tgt_lang = target_lang

max_length = 128
def preprocess(examples):
    # Get raw sentences
    inputs = examples["sinhala"]
    targets = examples["english"]
    
    # Tokenize source and target
    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        padding="max_length",
        truncation=True
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_length,
            padding="max_length",
            truncation=True
        )
    
    # Put labels in model_inputs
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs
# def preprocess(examples):
#     inputs = examples["sinhala"]
#     targets = examples["english"]
#     model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)
#     labels = tokenizer(targets, max_length=max_length, truncation=True)
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
eval_dataset = eval_dataset.map(preprocess, batched=True)


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)


Map:   0%|          | 0/3881 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/971 [00:00<?, ? examples/s]

In [10]:

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_sinhala2english",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=3e-4,
    num_train_epochs=10,
    logging_strategy="steps",
    eval_strategy="steps",
    save_strategy="steps",
    logging_dir="./logs",
    predict_with_generate=True,
    fp16=True,
    gradient_accumulation_steps=2,      
    dataloader_pin_memory=True,
    report_to="none",                   
    load_best_model_at_end=True,
    metric_for_best_model="loss",      
    save_total_limit=3,    
)

# Load BLEU metric
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    result["bleu"] = result["score"]
    return result

In [11]:
class RealTimeProgressCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        logs = kwargs.get("logs", {})
        if "loss" in logs:
            current_time = time.strftime("%H:%M:%S", time.localtime())
            print(f"[{current_time}] Step: {state.global_step}, Loss: {logs['loss']:.4f}", flush=True)
        if state.global_step % 3880 == 0:
            trainer.save_model(f"./nllb_sinhala2english_gpu/checkpoint-{state.global_step}")




trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
# Add the callback to the trainer
trainer.add_callback(RealTimeProgressCallback)

trainer.args.logging_steps = 1 
trainer.args.eval_steps = 3880
trainer.train()

model.save_pretrained("./nllb_sinhala2english_lora")
tokenizer.save_pretrained("./nllb_sinhala2english_lora")


/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss,Validation Loss,Score,Counts,Totals,Precisions,Bp,Sys Len,Ref Len,Bleu
3880,7.904500,7.457946,36.972585,"[13185, 8100, 5361, 3631]","[19296, 18325, 17361, 16410]","[68.33022388059702, 44.2019099590723, 30.879557629168826, 22.126751980499694]",0.975482,19296,19775,36.972585
7760,7.141200,7.447750,40.106330,"[13549, 8656, 5925, 4142]","[19292, 18321, 17356, 16406]","[70.2311839104292, 47.24632934883467, 34.138050241991245, 25.246860904547116]",0.975275,19292,19775,40.106330
11640,7.434500,7.441919,41.561060,"[13817, 8918, 6184, 4375]","[19597, 18626, 17662, 16713]","[70.50568964637445, 47.87930849350371, 35.013022307779416, 26.177227308083527]",0.990958,19597,19775,41.561060
15520,7.132000,7.443879,42.389038,"[13932, 9066, 6321, 4513]","[19525, 18554, 17589, 16640]","[71.35467349551857, 48.862778915597715, 35.93723349820911, 27.12139423076923]",0.987278,19525,19775,42.389038
19400,7.896700,7.448076,42.817915,"[13966, 9118, 6405, 4602]","[19567, 18596, 17631, 16682]","[71.37527469719426, 49.03204990320499, 36.32805853326527, 27.58662030931543]",0.989426,19567,19775,42.817915


('./nllb_sinhala2english_lora/tokenizer_config.json',
 './nllb_sinhala2english_lora/special_tokens_map.json',
 './nllb_sinhala2english_lora/sentencepiece.bpe.model',
 './nllb_sinhala2english_lora/added_tokens.json',
 './nllb_sinhala2english_lora/tokenizer.json')

In [ ]:
!zip -r nllb_sinhala2english_lora.zip ./nllb_sinhala2english_lora